# Orbit Wars: RL Training Pipeline

This notebook trains an RL policy ranker on Kaggle GPU.

**Workflow:**
1. Load previous weights from Kaggle Dataset (if continuing training)
2. Run REINFORCE training loop
3. Save weights as output
4. Download weights.npz and bundle with main_rl.py for submission

In [ ]:
# Cell 1: Environment setup
!pip install 'kaggle-environments>=1.28.0' -q
print('Environment ready.')

In [ ]:
# Cell 2: Write train_rl.py (the training script)
# Upload train_rl.py as a dataset or paste it here
import os

# Option A: If train_rl.py is uploaded as a Kaggle dataset
DATASET_PATH = '/kaggle/input/orbit-wars-weights'
WEIGHTS_FILE = os.path.join(DATASET_PATH, 'weights.npz') if os.path.exists(DATASET_PATH) else None

# Check if we have previous weights
if WEIGHTS_FILE and os.path.exists(WEIGHTS_FILE):
    print(f'Found previous weights at {WEIGHTS_FILE}')
else:
    WEIGHTS_FILE = None
    print('No previous weights found. Training from scratch.')

In [ ]:
# Cell 3: Import training code
# If train_rl.py is in a dataset, copy it to working dir
TRAIN_SCRIPT = '/kaggle/input/orbit-wars-code/train_rl.py'
if os.path.exists(TRAIN_SCRIPT):
    !cp {TRAIN_SCRIPT} /kaggle/working/train_rl.py

# Write inline if not available as dataset
if not os.path.exists('/kaggle/working/train_rl.py'):
    print('ERROR: Upload train_rl.py as a Kaggle dataset')
else:
    from train_rl import train_loop, NumpyMLP
    print('Training code loaded.')

In [ ]:
# Cell 4: Run training
SAVE_PATH = '/kaggle/working/weights.npz'

model, rewards = train_loop(
    n_episodes=200,
    lr=1e-3,
    weights_path=WEIGHTS_FILE,
    save_path=SAVE_PATH,
)

print(f'\nTraining done. Weights saved to {SAVE_PATH}')

In [ ]:
# Cell 5: Plot training progress
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(rewards, alpha=0.3, label='raw')
window = min(20, len(rewards))
if len(rewards) >= window:
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    plt.plot(range(window-1, len(rewards)), smoothed, label=f'avg({window})')
plt.xlabel('Episode'); plt.ylabel('Reward'); plt.legend(); plt.title('Training Rewards')

plt.subplot(1, 2, 2)
plt.hist(rewards, bins=30)
plt.xlabel('Reward'); plt.ylabel('Count'); plt.title('Reward Distribution')
plt.tight_layout()
plt.savefig('/kaggle/working/training_progress.png')
plt.show()

In [ ]:
# Cell 6: Test trained agent vs random
from train_rl import make_rl_agent
from kaggle_environments import make

rl_agent = make_rl_agent(model)

wins, total = 0, 10
for i in range(total):
    env = make('orbit_wars', debug=False)
    env.run([rl_agent, 'random'])
    r0 = env.steps[-1][0].reward or 0
    r1 = env.steps[-1][1].reward or 0
    if r0 > r1: wins += 1
    print(f'Game {i+1}: {r0:.0f} vs {r1:.0f} ({"WIN" if r0>r1 else "LOSS"})')

print(f'\nWin rate: {wins}/{total} = {100*wins/total:.0f}%')

In [ ]:
# Cell 7: Bundle for submission
# Copy main_rl.py as main.py and bundle with weights
!cp /kaggle/input/orbit-wars-code/main_rl.py /kaggle/working/main.py
!cd /kaggle/working && tar -czf submission.tar.gz main.py weights.npz
print('Created submission.tar.gz — download and submit!')
print('Command: kaggle competitions submit orbit-wars -f submission.tar.gz -m "RL agent"')